[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/ligo/ligo.ipynb)

# Gravitational-Wave Antenna Patterns

This example computes how strongly an L-shaped gravitational-wave detector such as LIGO responds to a wave from each direction on the sky. LIGO does not point at the sky: it hears waves from nearly every direction at once, but not equally well. A passing wave stretches space along one direction across its path and squeezes it along the other, and the detector reads how much more one arm lengthens than the other. The two LIGO detectors, at Hanford and Livingston, are compared over the whole sky, and a turning polarization shows that the wave has spin two.

A wave's strain is a map on offsets, `Strain = Point <- Point`: symmetric, without trace, and zero along the wave. An arm lengthens by the fraction `(~arm | strain(arm)) / 2`, the strain's quadratic form read along the arm, an arm's direction squaring to minus its length squared. The plus and cross strains are built from the two planes facing across the wave, each point moved by its height above them, as `east.dual() * (east & Point) - north.dual() * (north & Point)` and its counterpart, and a rotor turns those planes and the strain with them. One rotor per place on Earth, or per direction on the sky, gives a detector's level plane and the axes across a wave alike.

In the tensor notation of gravitational-wave astronomy the response reads as $F = D^{ij} e_{ij}$, with the detector tensor $D = \tfrac12(\hat a \otimes \hat a - \hat b \otimes \hat b)$ and $e$ the polarization tensor of the wave.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.relativity.ligo import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Offsets and directions are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Rotor = ga.gatype.rotor()
# Fractional stretch per offset.
Strain = ga.gatype((Point, Point))                         # Point <- Point

## 1. One frame for places and directions

The rotor that turns the first axis to the point at a latitude and longitude also turns the second axis to east and the third to north there: a turn from the equator towards the pole, then about the pole. At a detector's site it gives the level plane the arms lie in, each arm north turned towards east by its azimuth. For a direction on the sky it gives two axes across a wave from there. The algebra's axes are fixed to the Earth: x through longitude zero on the equator, y through ninety degrees east on it, z through the north pole.

In [ ]:
# The detectors: their names, latitudes and longitudes in degrees, and the azimuths of their two arms,
# in degrees clockwise from north.
names = ["Hanford", "Livingston"]
site_latitudes = [46.4553, 30.5631]
site_longitudes = [-119.4078, -90.7744]
azimuths = [[324, 234], [252, 162]]
# The sky, by the point on Earth each direction is straight above: latitudes from pole to pole, and twice
# as many longitudes once around.
n_sky_samples = 61

In [ ]:
def place(latitudes: np.ndarray, longitudes: np.ndarray) -> Rotor:
    """The frame at each latitude and longitude, in radians: first axis to the point, second to east, third to north."""
    return (mv.xy * (-longitudes / 2)).exp() * (mv.xz * (-latitudes / 2)).exp()


def arms(sites: Rotor, azimuths: np.ndarray) -> Point:
    """The arms of detectors at their sites, from their azimuths in radians, clockwise from north."""
    east, north = sites[..., None] >> mv.y, sites[..., None] >> mv.z          # [..., 1] Plane each
    return ((north ^ east) * (-azimuths / 2)).exp() >> north.dual()          # [..., 2] Point


sites = place(np.radians(site_latitudes), np.radians(site_longitudes))      # [detectors] Rotor
detector_arms = arms(sites, np.radians(azimuths))                            # [detectors, 2] Point
latitudes = np.radians(np.linspace(-90, 90, n_sky_samples))
longitudes = np.radians(np.linspace(-180, 180, 2 * n_sky_samples - 1))
sky = place(latitudes[:, None], longitudes)                                  # [latitudes, longitudes] Rotor
directions, zeniths = sky >> mv.x.dual(), sites >> mv.x.dual()               # [latitudes, longitudes], [detectors] Point

In [ ]:
detector_arms

## 2. Strains and the arms' reading

A wave's strain is a map on offsets: symmetric, without trace, and zero along the wave. For each direction, two strains span them all: plus, stretching along one axis across the wave and squeezing along the other, and cross, the same turned by an eighth of a turn. An arm along the unit direction `arm` lengthens by the fraction `(~arm | strain(arm)) / 2`, and the interferometer reads the difference between its two arms. The responses to plus and to cross are the detector's antenna patterns, and their root sum of squares, which does not depend on how the axes across the wave are turned, is its sensitivity. It is largest straight overhead and underfoot, and zero for waves arriving in the plane of the arms along the lines halfway between them: four blind spots on the sky. The first figure has a row per detector, its response to plus, to cross, and its sensitivity, each over the sky drawn above the point on Earth it is straight over: red where the arms read a wave positive and blue where they read it negative, and the sensitivity bright where it is largest and black at the blind spots.

The two LIGO detectors were built three thousand kilometres apart but nearly opposite in orientation, so that they hear nearly the same parts of the sky equally well and a real wave shows up in both at once: over most of the sky they read it with the same size and opposite signs. The second figure shows the two together: the network's sensitivity, the root sum of squares of the two, and how alike their readings of the same wave are, the cosine between their pairs of responses, blue where they read it with opposite signs and red where they read it with the same.

In [ ]:
def strains(frames: Rotor) -> tuple[Strain, Strain]:
    """The plus and cross strains of waves from each frame's first axis, with its other axes across the wave."""
    # The planes facing east and north across the wave; each point's height above them.
    east, north = frames >> mv.y, frames >> mv.z                             # [...] Plane each
    plus = east.dual() * (east & Point) - north.dual() * (north & Point)      # [...] Point <- Point
    cross = east.dual() * (north & Point) + north.dual() * (east & Point)     # [...] Point <- Point
    return plus, cross


def response(arms: Point, strain: Strain) -> Scalar:
    """The interferometer's reading: the difference between its two arms' stretches."""
    # A direction squares to minus its length squared, so the metric reads one along another by its reverse.
    stretches = (~arms | strain[..., None](arms)) / 2                         # [..., 2] Scalar
    return stretches[..., 0] - stretches[..., 1]                             # [...] Scalar


def sensitivity(plus: Scalar, cross: Scalar) -> Scalar:
    """The root sum of squares of the responses: the same for every turn of the axes across the wave."""
    return (plus.squared() + cross.squared()).square_root()


plus, cross = strains(sky)                                                   # [latitudes, longitudes] Point <- Point each
to_plus = response(detector_arms[:, None, None], plus)                       # [detectors, latitudes, longitudes] Scalar
to_cross = response(detector_arms[:, None, None], cross)                     # [detectors, latitudes, longitudes] Scalar
sensitivities = sensitivity(to_plus, to_cross)                               # [detectors, latitudes, longitudes] Scalar
network = sensitivities.squared().sum(axis=0).square_root()                  # [latitudes, longitudes] Scalar
# The cosine between the two detectors' pairs of responses: minus one where they read opposite signs.
alike = (to_plus[0] * to_plus[1] + to_cross[0] * to_cross[1]) / (sensitivities[0] * sensitivities[1])
save_figure(render.draw_patterns(directions, to_plus, to_cross, sensitivities, zeniths, names), "ligo_patterns")
save_figure(render.draw_network(directions, network, alike, zeniths, names), "ligo_network")

## 3. Spin two

Turn the axes across the wave about its direction by an angle, and plus turns into cross after an eighth of a turn and back into minus itself after a quarter: the strain repeats every half turn, the mark of a spin-two field. A ring of free masses across a wave from straight overhead shows it, on the left seen from above with the detector's arms, stretched into an ellipse that turns at the polarization angle, while the response over the whole sky, on the right, mixes plus and cross by twice that angle.

In [ ]:
# The polarization's half turn in frames, the free masses on the ring, and the strain drawn on the
# ring, far beyond any real one so that it shows; and the duration of each frame.
n_frames = 48
n_masses = 24
strain_shown = 0.3
duration_ms = 80

In [ ]:
def polarized(frames: Rotor, angles: np.ndarray) -> Rotor:
    """The frames turned about their first axis, the wave's direction, by the polarization angles."""
    return frames * (mv.yz * (-angles / 2)).exp()


angles = np.linspace(0, np.pi, n_frames, endpoint=False)                     # [frames]
turned_plus, _ = strains(polarized(sky, angles[:, None, None]))              # [frames, latitudes, longitudes] Point <- Point
hanford = response(detector_arms[0], turned_plus)                            # [frames, latitudes, longitudes] Scalar
# Straight overhead, the axes across the wave are the site's east and north.
east, north = sites[0] >> mv.y, sites[0] >> mv.z                             # [] Plane each
turns = np.linspace(0, 2 * np.pi, n_masses, endpoint=False)                  # [masses]
ring = ((east ^ north) * (-turns / 2)).exp() >> east.dual()                  # [masses] Point
overhead, _ = strains(polarized(sites[0], angles))                           # [frames] Point <- Point
rings = ring + strain_shown * overhead[:, None](ring)                         # [frames, masses] Point
show_animation(render.animate_polarization(directions, hanford, sites[0], detector_arms[0], rings, names[0]),
               "ligo_polarization", duration_ms)

In [ ]:
# checks
# Straight overhead each detector the sensitivity is one, and nowhere is it more.
overhead_sensitivity = sensitivity(*(response(detector_arms, strain) for strain in strains(sites)))
assert np.abs(render.scalars(overhead_sensitivity) - 1).max() < 1e-12
assert render.scalars(sensitivities).max() <= 1 + 1e-12
# A wave along the line halfway between the arms, in their plane, is not seen at all; any frame with
# its first axis along that line will do.
halfway = (detector_arms[:, 0] + detector_arms[:, 1]).normalized()          # [detectors] Point
blind = sensitivity(*(response(detector_arms, strain) for strain in strains((1 + halfway * ~mv.x.dual()).normalized())))
assert np.abs(render.scalars(blind)).max() < 1e-12
# Spin two: the turned response mixes plus and cross by twice the angle.
mixed = to_plus[0] * np.cos(2 * angles)[:, None, None] + to_cross[0] * np.sin(2 * angles)[:, None, None]
assert np.abs(render.scalars(hanford - mixed)).max() < 1e-12
# Over most of the sky the two detectors read a wave with opposite signs.
assert np.mean(render.scalars(alike) < -0.5) > 0.9